# Tests infrastructure DMR

In [2]:
import json

import pandas as pd
import geopandas as gpd

In [3]:
stations_pools_str = pd.read_csv("stations_pools_2026-07-25.csv")["extras"][0]
stations_pools = pd.DataFrame(json.loads(stations_pools_str))

stations_pdcs_str = pd.read_csv("stations_pdcs_2026-07-25.csv")["extras"][0]
stations_pdcs = pd.DataFrame(json.loads(stations_pdcs_str))

aires_dmr_gpd = gpd.read_file("aires_dmr_2026-07-08.geojson").rename(columns={"id_aire": "id_pool"})
aires_dmr = pd.DataFrame(aires_dmr_gpd[['Aire', 'NomUsuel', 'Nature', 'GVO', 'Surface', 'Réseau', 'Axe', 'GVO_Axe', 'id_pool']])


In [4]:
stations_pools

,id_pool,id_station_itinerance
0,A000001,FRHPCPNF080371TIERSTOTEM
1,A000002,FRTSLP5670
2,A000002,FRIOYP13531046
3,A000003,FRIOYP13530804
4,A000004,FRFASP11568703
...,...,...
537,A001685,FRIOYP13531103
538,A001698,FRTSLP30256
539,A001698,FRLDLPLFR3273EVCP
540,A001701,FRLDLPLFR3749EVCP


In [5]:
pools_pdcs = stations_pdcs.merge(right=stations_pools, on="id_station_itinerance")
aires_pdcs = aires_dmr.merge(right=pools_pdcs, on="id_pool", how="left")
aires_stations = aires_dmr.merge(right=stations_pools, on="id_pool", how="left")

In [15]:
aires_pdcs.to_csv("aires_pdc_2026-07-25.csv")
aires_dmr.to_csv("aires_dmr_2026-07-25.csv")

In [7]:
aires_stations_full = aires_pdcs.groupby(['id_station_itinerance']).agg(
    aire=pd.NamedAgg("Aire", "first"),
    nom=pd.NamedAgg("NomUsuel", "first"),
    nature=pd.NamedAgg("Nature", "first"),
    gvo=pd.NamedAgg("GVO", "first"),
    reseau=pd.NamedAgg("Réseau", "first"),
    axe=pd.NamedAgg("Axe", "first"),
    gvo_axe=pd.NamedAgg("GVO_Axe", "first"),
    id_pool=pd.NamedAgg("id_pool", "first"),
    latitude=pd.NamedAgg("latitude", "first"),
    longitude=pd.NamedAgg("longitude", "first"),
    puissance_cumul=pd.NamedAgg("puissance_nominale", "sum"),
).reset_index()
aires_stations_gpd=gdf = gpd.GeoDataFrame(
    aires_stations_full,
    geometry=gpd.points_from_xy(aires_stations_full["longitude"], aires_stations_full["latitude"]),
    crs="EPSG:4326"
)

In [14]:
aires_stations_gpd.to_csv("aires_stations.csv")

In [8]:
import folium

map = aires_dmr_gpd.explore(popup=True, name="aires", tiles="cartodbpositron")
aires_stations_gpd.explore(m=map, popup=True, name="stations", color="red")
folium.TileLayer().add_to(map)
folium.LayerControl().add_to(map)
map.save("aires_stations.html")
map

In [9]:
map = aires_stations_gpd.explore(
    popup=True,
    name="stations",
    tiles="OpenStreetMap",
)

aires_dmr_gpd.explore(
    m=map,
    name="aires",
)

folium.LayerControl().add_to(map)
map

In [10]:
aires_stations_gpd

,id_station_itinerance,aire,nom,nature,gvo,reseau,axe,gvo_axe,id_pool,latitude,longitude,puissance_cumul,geometry
0,FRA79P90932651,RRN NC,Centre routier Maison Blanche,S,DIRA,NC,N0010,NC,A001647,46.133704,0.178913,122.08,POINT (0.17891 46.1337)
1,FRALLPGO000081,RRN C,Aire de la Bruche,S,ARCOS,C,A0355,ARCOS,A001005,48.539027,7.580188,1822.00,POINT (7.58019 48.53903)
2,FRALLPGO000188,RRN C,TOULOUSE SUD NORD,S,ASF,C,A0061,ASF,A000648,43.485788,1.549115,3050.00,POINT (1.54912 43.48579)
3,FRALLPGO000216,RRN C,ROUILLE PAMPROUX NORD,S,ASF,C,A0010,ASF,A000378,46.453132,-0.017796,3000.00,POINT (-0.0178 46.45313)
4,FRALLPGO000226,RRN C,CHAVAGNES-EN-PAILLERS,S,ASF,C,A0083,ASF,A000142,46.873282,-1.283041,3050.00,POINT (-1.28304 46.87328)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
537,FRZUNP2771348324096144223,RRN C,Aire du pays d'Argentan,S,ALICORNE,C,A0088,ALICORNE,A000717,48.722935,-0.056772,2488.00,POINT (-0.05677 48.72293)
538,FRZUNP4015250050291921289,RRN C,Ocean Est,S,ATLANDES,C,A0063,ATLANDES,A000322,43.937372,-1.089300,6544.00,POINT (-1.0893 43.93737)
539,FRZUNP4029550098338231067,RRN C,Porte des Landes Ouest,S,ATLANDES,C,A0063,ATLANDES,A000727,44.361348,-0.852195,6554.20,POINT (-0.85219 44.36135)
540,FRZUNP5909246146811129817,RRN C,Porte des Landes Est,S,ATLANDES,C,A0063,ATLANDES,A000370,44.361107,-0.849591,6550.80,POINT (-0.84959 44.36111)


In [11]:
aires_pdcs

,Aire,NomUsuel,Nature,GVO,Surface,Réseau,Axe,GVO_Axe,id_pool,latitude,longitude,id_pdc_itinerance,puissance_nominale,id_station_itinerance
0,RRN C,Aire de Provence Verdon - VIDAUBAN-NORD,S,ESCOTA,24266.23969,C,A0008,ESCOTA,A000001,43.4155,6.4519,FRHPCENF080371013,150.0,FRHPCPNF080371TIERSTOTEM
1,RRN C,Aire de Provence Verdon - VIDAUBAN-NORD,S,ESCOTA,24266.23969,C,A0008,ESCOTA,A000001,43.4155,6.4519,FRHPCENF080371009,300.0,FRHPCPNF080371TIERSTOTEM
2,RRN C,Aire de Provence Verdon - VIDAUBAN-NORD,S,ESCOTA,24266.23969,C,A0008,ESCOTA,A000001,43.4155,6.4519,FRHPCENF080371002,150.0,FRHPCPNF080371TIERSTOTEM
3,RRN C,Aire de Provence Verdon - VIDAUBAN-NORD,S,ESCOTA,24266.23969,C,A0008,ESCOTA,A000001,43.4155,6.4519,FRHPCENF080371012,150.0,FRHPCPNF080371TIERSTOTEM
4,RRN C,Aire de Provence Verdon - VIDAUBAN-NORD,S,ESCOTA,24266.23969,C,A0008,ESCOTA,A000001,43.4155,6.4519,FRHPCENF080371008,300.0,FRHPCPNF080371TIERSTOTEM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7112,RRN NC,Aire de Beaumetz Les Loges,R,DIRN,2544.28482,NC,N0025,NC,A001871,NaN,NaN,NaN,NaN,NaN
7113,RRN NC,La Couette,S,DIRNO,12400.51192,NC,N0012,NC,A001872,NaN,NaN,NaN,NaN,NaN
7114,RRN NC,Le Bois de Vert,S,DIRNO,13021.44468,NC,N0012,NC,A001873,NaN,NaN,NaN,NaN,NaN
7115,RRN NC,Les Veys,R,DIRNO,29190.57638,NC,N0013,NC,A001874,NaN,NaN,NaN,NaN,NaN


In [12]:
aires_stations[:]

,Aire,NomUsuel,Nature,GVO,Surface,Réseau,Axe,GVO_Axe,id_pool,id_station_itinerance
0,RRN C,Aire de Provence Verdon - VIDAUBAN-NORD,S,ESCOTA,24266.23969,C,A0008,ESCOTA,A000001,FRHPCPNF080371TIERSTOTEM
1,RRN C,VIDAUBAN-SUD,S,ESCOTA,58303.58781,C,A0008,ESCOTA,A000002,FRTSLP5670
2,RRN C,VIDAUBAN-SUD,S,ESCOTA,58303.58781,C,A0008,ESCOTA,A000002,FRIOYP13531046
3,RRN C,ROUSSET,S,ESCOTA,51589.58553,C,A0008,ESCOTA,A000003,FRIOYP13530804
4,RRN C,MANOIRE,S,ASF,32668.21678,C,A0089,ASF,A000004,FRFASP11568703
...,...,...,...,...,...,...,...,...,...,...
1921,RRN NC,Aire de Beaumetz Les Loges,R,DIRN,2544.28482,NC,N0025,NC,A001871,NaN
1922,RRN NC,La Couette,S,DIRNO,12400.51192,NC,N0012,NC,A001872,NaN
1923,RRN NC,Le Bois de Vert,S,DIRNO,13021.44468,NC,N0012,NC,A001873,NaN
1924,RRN NC,Les Veys,R,DIRNO,29190.57638,NC,N0013,NC,A001874,NaN
